In [8]:
# Conversation Management & Classification using Groq API
# Updated with Current Supported Models (September 2024)

import json
import openai
from typing import List, Dict, Any, Optional
from datetime import datetime
import re

# ===== SETUP AND CONFIGURATION =====

class GroqConversationManager:
    """
    Manages conversation history with summarization capabilities using Groq API
    """

    def __init__(self, api_key: str, model: str = "llama-3.1-70b-versatile"):  # Updated model
        """
        Initialize the conversation manager

        Args:
            api_key: Your Groq API key
            model: Model to use for completions
        """
        self.client = openai.OpenAI(
            base_url="https://api.groq.com/openai/v1",
            api_key=api_key
        )
        self.model = model
        self.conversation_history = []
        self.summary_history = []
        self.conversation_count = 0

    def add_message(self, role: str, content: str):
        """Add a message to conversation history"""
        message = {
            "role": role,
            "content": content,
            "timestamp": datetime.now().isoformat()
        }
        self.conversation_history.append(message)
        self.conversation_count += 1

    def get_conversation_length(self) -> Dict[str, int]:
        """Get conversation statistics"""
        total_chars = sum(len(msg["content"]) for msg in self.conversation_history)
        total_words = sum(len(msg["content"].split()) for msg in self.conversation_history)
        return {
            "turns": len(self.conversation_history),
            "characters": total_chars,
            "words": total_words
        }

    def truncate_by_turns(self, max_turns: int) -> List[Dict]:
        """Truncate conversation to last n turns"""
        if max_turns >= len(self.conversation_history):
            return self.conversation_history.copy()
        return self.conversation_history[-max_turns:]

    def truncate_by_length(self, max_chars: int = None, max_words: int = None) -> List[Dict]:
        """Truncate conversation by character or word limit"""
        if not max_chars and not max_words:
            return self.conversation_history.copy()

        truncated = []
        current_chars = 0
        current_words = 0

        # Go backwards through conversation
        for msg in reversed(self.conversation_history):
            msg_chars = len(msg["content"])
            msg_words = len(msg["content"].split())

            if max_chars and (current_chars + msg_chars) > max_chars:
                break
            if max_words and (current_words + msg_words) > max_words:
                break

            truncated.insert(0, msg)
            current_chars += msg_chars
            current_words += msg_words

        return truncated

    def summarize_conversation(self, messages: List[Dict]) -> str:
        """Generate summary of conversation using Groq API"""
        if not messages:
            return "No conversation to summarize."

        # Prepare conversation text
        conv_text = "\n".join([f"{msg['role']}: {msg['content']}" for msg in messages])

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {
                        "role": "system",
                        "content": "You are a helpful assistant that creates concise summaries of conversations. Summarize the key points, topics discussed, and any important information exchanged."
                    },
                    {
                        "role": "user",
                        "content": f"Please summarize this conversation:\n\n{conv_text}"
                    }
                ],
                temperature=0.3,
                max_tokens=500
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"Error generating summary: {str(e)}"

    def periodic_summarization(self, k_runs: int = 3):
        """Perform summarization after every k-th conversation turn"""
        if self.conversation_count % k_runs == 0 and self.conversation_count > 0:
            print(f"\n🔄 Performing periodic summarization after {self.conversation_count} turns...")

            # Summarize current conversation
            summary = self.summarize_conversation(self.conversation_history)

            # Store summary
            summary_entry = {
                "summary": summary,
                "original_turns": len(self.conversation_history),
                "timestamp": datetime.now().isoformat(),
                "conversation_range": f"Turns 1-{len(self.conversation_history)}"
            }
            self.summary_history.append(summary_entry)

            # Replace conversation history with summary (keeping last 2 messages for context)
            context_messages = self.conversation_history[-2:] if len(self.conversation_history) >= 2 else self.conversation_history

            # Create new conversation with summary + context
            self.conversation_history = [
                {
                    "role": "system",
                    "content": f"Previous conversation summary: {summary}",
                    "timestamp": datetime.now().isoformat()
                }
            ] + context_messages

            print(f"✅ Summary created and conversation compressed from {summary_entry['original_turns']} to {len(self.conversation_history)} messages")
            return summary

        return None

# ===== TASK 2: JSON SCHEMA CLASSIFICATION & EXTRACTION =====

class InformationExtractor:
    """
    Extracts structured information from chats using Groq API
    """

    def __init__(self, api_key: str, model: str = "llama-3.1-70b-versatile"):  # Updated model
        self.client = openai.OpenAI(
            base_url="https://api.groq.com/openai/v1",
            api_key=api_key
        )
        self.model = model

        # Define JSON schema for information extraction
        self.extraction_schema = {
            "type": "object",
            "properties": {
                "name": {
                    "type": ["string", "null"],
                    "description": "Full name of the person"
                },
                "email": {
                    "type": ["string", "null"],
                    "description": "Email address"
                },
                "phone": {
                    "type": ["string", "null"],
                    "description": "Phone number"
                },
                "location": {
                    "type": ["string", "null"],
                    "description": "Location, city, or address"
                },
                "age": {
                    "type": ["integer", "null"],
                    "description": "Age in years"
                }
            }
        }

        # Define function for structured extraction
        self.extraction_function = {
            "name": "extract_user_information",
            "description": "Extract user information from chat conversation",
            "parameters": self.extraction_schema
        }

    def extract_information(self, chat_text: str) -> Dict[str, Any]:
        """Extract structured information from chat using function calling"""
        try:
            # Use structured prompting approach first
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {
                        "role": "system",
                        "content": "You are an expert information extraction assistant. Extract user information from conversations. Return a valid JSON object with fields: name, email, phone, location, age. Use null for missing fields."
                    },
                    {
                        "role": "user",
                        "content": f"""Extract information from this chat conversation and return ONLY a JSON object:

{chat_text}

Return JSON format:
{{"name": "extracted name or null", "email": "extracted email or null", "phone": "extracted phone or null", "location": "extracted location or null", "age": extracted_age_number_or_null}}"""
                    }
                ],
                temperature=0.1,
                max_tokens=200
            )

            # Parse JSON response
            response_text = response.choices[0].message.content.strip()

            # Clean response text to extract JSON
            if "```json" in response_text:
                json_start = response_text.find("```json") + 7
                json_end = response_text.find("```", json_start)
                json_text = response_text[json_start:json_end]
            elif "{" in response_text and "}" in response_text:
                json_start = response_text.find("{")
                json_end = response_text.rfind("}") + 1
                json_text = response_text[json_start:json_end]
            else:
                json_text = response_text

            extracted_data = json.loads(json_text)
            return self.validate_extraction(extracted_data)

        except Exception as e:
            print(f"LLM extraction failed: {e}")
            # Use enhanced fallback regex extraction
            return self.enhanced_fallback_extraction(chat_text)

    def enhanced_fallback_extraction(self, chat_text: str) -> Dict[str, Any]:
        """Enhanced fallback extraction using improved regex patterns"""
        extracted = {}

        # Clean the text for better processing
        text_clean = chat_text.replace('\n', ' ').strip()

        # Extract name with improved patterns
        name_patterns = [
            r"(?:name is|I'm|I am|my name's|my name is|call me)\s+([A-Za-z][A-Za-z\s]{1,30}?)(?:\s+and|\s*[,.]|$|\s+\d)",
            r"User:\s*([A-Za-z][A-Za-z\s]{1,30}?)(?:\s+and|\s*[,.]|$|\s+\d)",
            r"reservation under\s+([A-Za-z][A-Za-z\s]{1,30}?)(?:\s+and|\s*[,.]|$|\s+\d)"
        ]
        name = None
        for pattern in name_patterns:
            match = re.search(pattern, text_clean, re.IGNORECASE)
            if match:
                potential_name = match.group(1).strip()
                # Filter out common false positives
                if not re.search(r'\b(service|account|help|thanks|great|perfect)\b', potential_name, re.IGNORECASE):
                    name = potential_name
                    break

        # Extract email with strict pattern
        email_match = re.search(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,7}\b', text_clean)
        email = email_match.group(0) if email_match else None

        # Extract phone with multiple formats
        phone_patterns = [
            r'\b(\d{3}[-.\s]?\d{3}[-.\s]?\d{4})\b',
            r'\((\d{3})\)\s*(\d{3}[-.\s]?\d{4})',
            r'\b(\d{10})\b'
        ]
        phone = None
        for pattern in phone_patterns:
            matches = re.findall(pattern, text_clean)
            if matches:
                if isinstance(matches[0], tuple):
                    phone = ''.join(matches[0])
                else:
                    phone = matches[0]
                break

        # Extract location with improved patterns
        location_patterns = [
            r"(?:in|from|live in|based in|located in|live)\s+([A-Za-z\s,]+(?:California|CA|New York|NY|Chicago|IL|Texas|TX|Florida|FL|San Francisco|SF|Los Angeles|LA|Seattle|WA))",
            r"(?:in|from|live in|based in|located in|live)\s+([A-Za-z\s,]{2,25}?)(?:\s*[,.]|$|\s+and)",
            r"(?:downtown|in)\s+([A-Za-z]{2,15})\b"
        ]
        location = None
        for pattern in location_patterns:
            match = re.search(pattern, text_clean, re.IGNORECASE)
            if match:
                potential_location = match.group(1).strip()
                # Clean up common issues
                potential_location = re.sub(r'\s+', ' ', potential_location)
                if len(potential_location) >= 2 and not re.search(r'\b(years|old|email|phone)\b', potential_location, re.IGNORECASE):
                    location = potential_location
                    break

        # Extract age with improved patterns
        age_patterns = [
            r"(?:I'm|I am|age)\s*(\d{1,3})(?:\s*years?\s*old)?",
            r"(\d{1,3})\s*years?\s*old",
            r"(?:old|age)[\s,]+(\d{1,3})"
        ]
        age = None
        for pattern in age_patterns:
            match = re.search(pattern, text_clean, re.IGNORECASE)
            if match:
                age_val = int(match.group(1))
                if 10 <= age_val <= 120:  # Reasonable age range
                    age = age_val
                    break

        return self.validate_extraction({
            "name": name,
            "email": email,
            "phone": phone,
            "location": location,
            "age": age
        })

    def validate_extraction(self, data: Dict[str, Any]) -> Dict[str, Any]:
        """Validate extracted data against schema"""
        validated = {}

        # Validate each field
        for field in self.extraction_schema["properties"]:
            value = data.get(field)

            if value is not None and str(value).strip() != '' and str(value).lower() != 'null':
                if field == "email":
                    # Enhanced email validation
                    email_str = str(value).strip()
                    if "@" in email_str and "." in email_str.split("@")[-1]:
                        validated[field] = email_str
                    else:
                        validated[field] = None
                elif field == "age":
                    # Enhanced age validation
                    try:
                        age = int(value)
                        validated[field] = age if 10 <= age <= 120 else None
                    except:
                        validated[field] = None
                elif field == "name":
                    # Enhanced name validation
                    name_str = str(value).strip()
                    if len(name_str) >= 2 and not re.search(r'\d|@', name_str):
                        validated[field] = name_str
                    else:
                        validated[field] = None
                elif field == "phone":
                    # Clean phone number
                    phone_str = re.sub(r'[^\d]', '', str(value))
                    if len(phone_str) >= 10:
                        validated[field] = str(value).strip()
                    else:
                        validated[field] = None
                else:
                    # General validation
                    validated[field] = str(value).strip()
            else:
                validated[field] = None

        return validated

# ===== DEMONSTRATION FUNCTIONS =====

def demonstrate_task1():
    """Demonstrate Task 1: Conversation Management with Summarization"""
    print("=" * 60)
    print("TASK 1: CONVERSATION MANAGEMENT & SUMMARIZATION")
    print("=" * 60)

    # Use your actual Groq API key
    API_KEY = "YOUR_GROQ_API_KEY_HERE"  # Replace with your actual key
    conv_manager = GroqConversationManager(API_KEY)

    # Sample conversations
    sample_conversations = [
        ("user", "Hi, I'm looking for a good restaurant recommendation in New York."),
        ("assistant", "I'd be happy to help! What type of cuisine are you interested in?"),
        ("user", "I love Italian food, especially pasta dishes."),
        ("assistant", "Great choice! I recommend 'Carbone' in Greenwich Village. They serve excellent Italian-American cuisine with amazing pasta dishes."),
        ("user", "That sounds perfect! What's their price range?"),
        ("assistant", "Carbone is on the higher end, expect to spend around $80-120 per person. Would you like me to suggest some more affordable options too?"),
        ("user", "Yes, please give me a few more affordable alternatives."),
        ("assistant", "Sure! Here are some great affordable Italian options: 1) Joe's Pizza - famous NYC pizza, 2) Lombardi's - historic pizza place, 3) Pasta Flyer - great pasta under $20."),
        ("user", "Thanks! Can you also recommend some good coffee shops nearby?"),
        ("assistant", "Absolutely! Near those restaurants you'll find: Stumptown Coffee, Blue Bottle Coffee, and Joe Coffee - all excellent choices for quality coffee."),
    ]

    print("\n📝 Adding conversation messages...")
    for i, (role, content) in enumerate(sample_conversations, 1):
        conv_manager.add_message(role, content)
        print(f"Turn {i}: {role} - {content[:50]}...")

        # Demonstrate periodic summarization every 3 turns
        summary = conv_manager.periodic_summarization(k_runs=3)
        if summary and not summary.startswith("Error"):
            print(f"📋 Summary: {summary[:100]}...")

    # Demonstrate truncation options
    print(f"\n📊 Conversation Statistics:")
    stats = conv_manager.get_conversation_length()
    for key, value in stats.items():
        print(f"  {key.title()}: {value}")

    print(f"\n🔄 Demonstrating Truncation Options:")

    # Truncate by turns
    truncated_turns = conv_manager.truncate_by_turns(4)
    print(f"\nLast 4 turns: {len(truncated_turns)} messages")
    for msg in truncated_turns[-2:]:  # Show last 2
        print(f"  {msg['role']}: {msg['content'][:50]}...")

    # Truncate by characters
    truncated_chars = conv_manager.truncate_by_length(max_chars=200)
    print(f"\nLast 200 characters: {len(truncated_chars)} messages")

    # Truncate by words
    truncated_words = conv_manager.truncate_by_length(max_words=50)
    print(f"Last 50 words: {len(truncated_words)} messages")

    # Show summary history
    print(f"\n📚 Summary History ({len(conv_manager.summary_history)} summaries):")
    for i, summary in enumerate(conv_manager.summary_history, 1):
        if not summary['summary'].startswith("Error"):
            print(f"  Summary {i}: {summary['summary'][:80]}...")

def demonstrate_task2():
    """Demonstrate Task 2: JSON Schema Classification & Information Extraction"""
    print("\n" + "=" * 60)
    print("TASK 2: JSON SCHEMA CLASSIFICATION & INFORMATION EXTRACTION")
    print("=" * 60)

    # Use your actual Groq API key
    API_KEY = "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"  # Replace with your actual key
    extractor = InformationExtractor(API_KEY)

    # Sample chat conversations for information extraction
    sample_chats = [
        {
            "id": 1,
            "chat": """
            User: Hi, I'd like to sign up for your service.
            Assistant: Great! I'll need some information from you.
            User: Sure, my name is John Smith and I'm 28 years old.
            Assistant: Thanks John! What's your email address?
            User: It's john.smith@email.com
            Assistant: And your phone number?
            User: 555-123-4567. I live in Los Angeles, California.
            Assistant: Perfect! I have all the information I need.
            """
        },
        {
            "id": 2,
            "chat": """
            User: Hello, I need help with my account.
            Assistant: I'd be happy to help! Can you provide your details?
            User: My name is Sarah Johnson, email sarah.j@company.com
            Assistant: Thanks Sarah! What's your location?
            User: I'm based in Chicago, IL. I'm 35 years old.
            Assistant: Great! Do you have a phone number on file?
            User: Yes, it's (312) 555-9876
            """
        },
        {
            "id": 3,
            "chat": """
            User: I want to make a reservation.
            Assistant: Certainly! What name should I put the reservation under?
            User: Mike Chen
            Assistant: Perfect! Any contact information?
            User: You can reach me at mike.chen.dining@outlook.com or call 415-555-0123
            Assistant: Thanks! Are you local to San Francisco?
            User: Yes, I live in downtown SF. I'm 42.
            """
        }
    ]

    print(f"\n🔍 Extracting Information from {len(sample_chats)} Chat Samples:")
    print("-" * 50)

    results = []
    for chat_sample in sample_chats:
        print(f"\n📱 Chat Sample {chat_sample['id']}:")
        print(f"Chat Preview: {' '.join(chat_sample['chat'].split()[:15])}...")

        # Extract information
        extracted = extractor.extract_information(chat_sample['chat'])
        results.append({
            "chat_id": chat_sample['id'],
            "extracted_data": extracted
        })

        # Display results
        print(f"📋 Extracted Information:")
        if "error" in extracted:
            print(f"  ❌ Error: {extracted['error']}")
        else:
            for field, value in extracted.items():
                status = "✅" if value else "❌"
                print(f"  {status} {field.title()}: {value}")

    # Validate against schema
    print(f"\n🔍 Schema Validation Summary:")
    print("-" * 30)

    for result in results:
        chat_id = result['chat_id']
        data = result['extracted_data']

        if "error" not in data:
            filled_fields = sum(1 for v in data.values() if v is not None)
            total_fields = len(extractor.extraction_schema['properties'])
            print(f"Chat {chat_id}: {filled_fields}/{total_fields} fields extracted")
        else:
            print(f"Chat {chat_id}: Extraction failed")

    return results

# ===== MAIN EXECUTION =====

def main():
    """Main execution function"""
    print("🚀 GROQ API ASSIGNMENT: CONVERSATION MANAGEMENT & CLASSIFICATION")
    print("Updated with Current Supported Models (September 2024)")
    print("=" * 70)

    print("✅ Using updated Groq model: llama-3.1-70b-versatile")
    print("📝 Remember to replace 'YOUR_GROQ_API_KEY_HERE' with your actual API key")
    print("-" * 70)

    try:
        # Run demonstrations
        demonstrate_task1()
        demonstrate_task2()

        print(f"\n🎉 Assignment completed successfully!")
        print("📝 Assignment Features Demonstrated:")
        print("  ✅ Conversation management with periodic summarization")
        print("  ✅ Multiple truncation methods (turns, characters, words)")
        print("  ✅ JSON schema-based information extraction")
        print("  ✅ LLM extraction with regex fallback")
        print("  ✅ Data validation and error handling")

    except Exception as e:
        print(f"❌ Error during execution: {str(e)}")
        print("Please check your API key and internet connection.")

if __name__ == "__main__":
    main()

🚀 GROQ API ASSIGNMENT: CONVERSATION MANAGEMENT & CLASSIFICATION
Updated with Current Supported Models (September 2024)
✅ Using updated Groq model: llama-3.1-70b-versatile
📝 Remember to replace 'YOUR_GROQ_API_KEY_HERE' with your actual API key
----------------------------------------------------------------------
TASK 1: CONVERSATION MANAGEMENT & SUMMARIZATION

📝 Adding conversation messages...
Turn 1: user - Hi, I'm looking for a good restaurant recommendati...
Turn 2: assistant - I'd be happy to help! What type of cuisine are you...
Turn 3: user - I love Italian food, especially pasta dishes....

🔄 Performing periodic summarization after 3 turns...
✅ Summary created and conversation compressed from 3 to 3 messages
Turn 4: assistant - Great choice! I recommend 'Carbone' in Greenwich V...
Turn 5: user - That sounds perfect! What's their price range?...
Turn 6: assistant - Carbone is on the higher end, expect to spend arou...

🔄 Performing periodic summarization after 6 turns...
✅ Summar